# 02 — EDA: Bank Marketing Kampagnendaten

**Person:** A/B | **Hypothesen:** H1, H2

**Ziel:**
- Datensatz-Struktur verstehen
- Response-Rate analysieren (Zielvariable `y`)
- Segment-relevante Features identifizieren

In [36]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from src.data.loader import load_bank_marketing, RAW_DIR
from src.data.preprocessor import BankMarketingPreprocessor
from src.utils.helpers import save_figure

print('Setup OK')

Setup OK


## 1. Daten laden

In [37]:
df_raw = pd.read_csv("../data/raw/bank_marketing/bank-additional-full.csv", sep=";")
df_raw.head()
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

## 2. Klassen-Verteilung (Zielvariable y)

In [38]:
print(df['y'].value_counts(normalize=True))

y
no     0.887346
yes    0.112654
Name: proportion, dtype: float64


## 3. Feature-Analyse

In [39]:
df.describe(include='all')
for col in ['job', 'education', 'marital', 'age']:
    response = df.groupby(col)['y'].value_counts(normalize=True).unstack()
    print(f'\n--- {col} ---')
    print(response['yes'].sort_values(ascending=False))



--- job ---
job
student          0.314286
retired          0.252326
unemployed       0.142012
admin.           0.129726
management       0.112175
unknown          0.112121
technician       0.108260
self-employed    0.104856
housemaid        0.100000
entrepreneur     0.085165
services         0.081381
blue-collar      0.068943
Name: yes, dtype: float64

--- education ---
education
illiterate             0.222222
unknown                0.145003
university.degree      0.137245
professional.course    0.113485
high.school            0.108355
basic.4y               0.102490
basic.6y               0.082024
basic.9y               0.078246
Name: yes, dtype: float64

--- marital ---
marital
unknown     0.150000
single      0.140041
divorced    0.103209
married     0.101573
Name: yes, dtype: float64

--- age ---
age
98    1.000000
89    1.000000
87    1.000000
92    0.750000
77    0.650000
        ...   
49    0.065554
47    0.062500
91         NaN
94         NaN
95         NaN
Name: yes, Length

## 4. Response-Rate nach demografischen Segmenten

In [40]:
for col in ['job', 'education', 'marital', 'age']:
     response = df_raw.groupby(col)['y'].value_counts(normalize=True).unstack()
     print(f'\n--- {col} ---')
     print(response['yes'].sort_values(ascending=False))


--- job ---
job
student          0.314286
retired          0.252326
unemployed       0.142012
admin.           0.129726
management       0.112175
unknown          0.112121
technician       0.108260
self-employed    0.104856
housemaid        0.100000
entrepreneur     0.085165
services         0.081381
blue-collar      0.068943
Name: yes, dtype: float64

--- education ---
education
illiterate             0.222222
unknown                0.145003
university.degree      0.137245
professional.course    0.113485
high.school            0.108355
basic.4y               0.102490
basic.6y               0.082024
basic.9y               0.078246
Name: yes, dtype: float64

--- marital ---
marital
unknown     0.150000
single      0.140041
divorced    0.103209
married     0.101573
Name: yes, dtype: float64

--- age ---
age
98    1.000000
89    1.000000
87    1.000000
92    0.750000
77    0.650000
        ...   
49    0.065554
47    0.062500
91         NaN
94         NaN
95         NaN
Name: yes, Length

🇩🇪 5. Key Findings
Basierend auf der durchgeführten EDA und den beobachteten Zusammenhängen im Datensatz:

Finding 1: Der Beruf hat einen starken Einfluss auf die Abschlusswahrscheinlichkeit
Die Analyse zeigt deutliche Unterschiede zwischen den Berufsgruppen.
Studierende (31,4%) und Rentner (25,2%) weisen die höchsten Abschlussraten für das Termingeldprodukt auf.
Berufsgruppen wie blue-collar (6,9%) und services (8,1%) zeigen dagegen deutlich niedrigere Werte.

➡️ Interpretation:  
Personen mit stabiler finanzieller Situation oder höherer Bildung reagieren stärker auf langfristige Sparprodukte, während körperlich arbeitende Berufsgruppen weniger affin sind.

Finding 2: Das Bildungsniveau korreliert mit der Abschlussrate
Kunden mit university degree (13,7%) oder professional course (11,3%) zeigen höhere Abschlusswahrscheinlichkeiten.
Die Kategorie illiterate fällt mit 22,2% besonders auf, obwohl sie sehr klein ist.
Niedrigere Bildungsstufen wie basic.4y oder high.school liegen im Mittelfeld.

➡️ Interpretation:  
Höhere Bildung geht tendenziell mit höherem Vertrauen in Finanzprodukte einher. Gleichzeitig zeigt die hohe Rate bei „illiterate“, dass bestimmte demografische Gruppen besonders empfänglich für Bankangebote sein können.

Finding 3: Der Datensatz ist vollständig und gut strukturiert
Alle 41.188 Beobachtungen sind vollständig, ohne fehlende Werte (NaN).
Die Variablen sind korrekt typisiert (kategorisch vs. numerisch).
Kategorien wie unknown müssen jedoch als fehlende Information interpretiert und entsprechend behandelt werden.

➡️ Interpretation:  
Der Datensatz eignet sich sehr gut für Modellierung und Machine Learning, benötigt aber eine saubere Vorverarbeitung (z. B. Umgang mit „unknown“, Skalierung numerischer Variablen).